# Reproduce the iMSE results

Open this notebook inside the repository. Select any of the three paper datasets.
`replay` reconstructs the tables from the released checkpoint metrics in seconds.
`all` downloads and recomputes the experiment. Use `import-cache` for the original
feature banks, then `evaluate` to score them. See [the protocol](../docs/protocol.md)
for ImageNet access, disk requirements and the Vim runtime.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO = Path.cwd()
if not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
assert (REPO / "src/imse/assets/datasets.json").is_file(), "Open the notebook from the repo"

DATASETS = ["CIFAR10"]  # also CIFAR100 and ImageNet1K
MODE = "replay"  # replay, all, download, extract, import-cache, evaluate
BACKBONE = None  # all paper backbones, or an exact backbone identifier
DEVICE = "cuda"
LEGACY_ROOT = None  # path to the original Colab Notebooks directory
WORK = REPO / "work"
SENSITIVITY = False
ABLATION = False
STRICT = True

# Set True once before GPU execution if the environment needs installation.
INSTALL = False
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO)], check=True)

In [ ]:
output = WORK / ("replay" if MODE == "replay" else "results")
args = [sys.executable, "-m", "imse", MODE, "--dataset", *DATASETS,
        "--data", str(WORK / "data"), "--weights", str(WORK / "weights"),
        "--features", str(WORK / "features"), "--output", str(output), "--device", DEVICE]
if BACKBONE:
    args += ["--backbone", BACKBONE]
if LEGACY_ROOT:
    args += ["--legacy-root", str(LEGACY_ROOT)]
if SENSITIVITY:
    args += ["--sensitivity"]
if ABLATION:
    args += ["--ablation"]
if STRICT:
    args += ["--require-paper-match"]
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src") + os.pathsep + env.get("PYTHONPATH", "")
subprocess.run(args, cwd=REPO, env=env, check=True)

In [ ]:
import pandas as pd

if MODE in {"replay", "evaluate", "all"}:
    sys.path.insert(0, str(REPO / "src"))
    from imse.benchmark.report import display_table

    pairs = pd.read_csv(output / "backbones.csv")
    average = pd.read_csv(output / "dataset_means.csv") if BACKBONE is None else None
    for dataset in DATASETS:
        for scope in ("All", "Near", "Far"):
            print(dataset, scope)
            print(display_table(pairs, average, dataset, scope).to_string())
    print("Reference comparison:", output / "reference_comparison.csv")